In [0]:
#import pyspark.sql.functions as F
from pyspark.sql.functions import col, lower, when, substring_index, regexp_replace, explode, split, hour
import os

In [0]:
table_path = "workspace.default.cloud_app_events_banking_dlp"
current_directory = os.getcwd() 
df = spark.table(table_path)

In [0]:
notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
repo_root = os.path.dirname(notebook_path)

#Get File Name and Extension 
- This Feature will be extracting file extension from file name. For DLP PDFs, CSVs, and Text files are higher risk than system files (like .exe) 
because they are the primary containers for sensitive client PII.

In [0]:
df_features = df.withColumn("file_name", regexp_replace(col("ObjectName"),r"\.[^.]+$", "")
                            ).withColumn("file_extension", substring_index(col("ObjectName"), ".", -1))
display(df_features.limit(100))


#Double Extension Detection:
- Identifies "masked" files ('data.csv.zip'). This flags attempts 
to bypass file-type filters by hiding sensitive extensions inside 
another filename.


In [0]:
extension_pattern = r"\.(pdf|docx|xlsx|doc|xls|ppt|pptx|csv|zip|7z|rar|py|sh|bat|ps1|sql|txt)"

df_features = df_features.withColumn(
    "double_extension_check", 
    when(col("file_name").rlike(extension_pattern), 1).otherwise(0)
)

  
                                  
display(df_features.limit(100))

## Risky Domains Flag:

* Checks the **Target Domain** and flags it if the domain exists in `risky_domains.csv`. Since these domains are not used in a standard business process, there is a higher risk of a data leak associated with them.

In [0]:

csv_path = os.path.join(current_directory, "risky_domains.csv")
risky_domains_df = spark.read.csv(csv_path, header=True, inferSchema=True)


In [0]:
risky_domains_df = risky_domains_df.withColumn(
    "Domains", 
    explode(split(col("Domain"), r",\s*"))
)

risky_domains_list = [row[0] for row in risky_domains_df.select("Domains").collect()]

regex_pattern = "|".join([d.replace(".", r"\.") for d in risky_domains_list])

df_features = df_features.withColumn(
    "is_risky_destination",
    when(lower(col("Target_Domain")).rlike(regex_pattern), 1).otherwise(0)
)

display(df_features)

# Flagging data exfiltration after work hours
After-work uploads could be considered more risky since there is a lower probability that they are work-related.


In [0]:
df_features = df_features.withColumn(
    "after_work_hour", hour(col("Timestamp"))
).withColumn(
    "is_after_hours", 
    when((col("after_work_hour") >= 19) | (col("after_work_hour") <= 5), 1).otherwise(0)
)

display(df_features)

#Clean Code Cell
with comments


In [0]:
from pyspark.sql.functions import col, lower, when, substring_index, regexp_replace, explode, split, hour

# Define extension pattern to match common file extensions
extension_pattern = r"\.(pdf|docx|xlsx|doc|xls|ppt|pptx|csv|zip|7z|rar|py|sh|bat|ps1|sql|txt)"

# Flagging data exfiltration risks by matching target domains against a dynamic blacklist of non-business sanctioned apps.

risky_domains_df = risky_domains_df.withColumn(
    "Domains", 
    explode(split(col("Domain"), r",\s*"))
)

risky_domains_list = [row[0] for row in risky_domains_df.select("Domains").collect()]
regex_pattern = "|".join([d.replace(".", r"\.") for d in risky_domains_list])




df_features_clean = df_features.withColumn(
    # Extracts everything after the last dot
    "file_extension", lower(substring_index(col("ObjectName"), ".", -1))
).withColumn(
    # Extracts everything BEFORE the last dot by searching for the last dot and replacing whats after with an empty string
    "file_name", regexp_replace(col("ObjectName"), r"\.[^.]+$", "")
).withColumn(
    # Check 1: Is there an extension left after removing one?
    "double_extension_check", 
    when(col("file_name").rlike(extension_pattern), 1).otherwise(0)
).withColumn(
    # Check 2: Is the risky target domain?
    "is_risky_target_domain",
    when(lower(col("Target_Domain")).rlike(regex_pattern), 1).otherwise(0)
).withColumn(
    # Check 3: if between 7 PM and 5 AM 
    "is_after_hours", 
    when((col("after_work_hour") >= 19) | (col("after_work_hour") <= 5), 1).otherwise(0)
)
display(df_features_clean.limit(100))
   

#Tests Below

In [0]:
display(df_features.filter(col("double_extension_check") == 1))

In [0]:
display(df_features.filter(df.ObjectName.contains(".pdf.")))

In [0]:
df_features.select("Target_Domain").distinct().show()